# Sultai bounded local search

The shared deck has 34 spells and 26 lands. Four Caverns, four Towns and two Cottages are fixed; Palace is capped at two and basics together at two. Three starts share cached components. Every score measures casting opportunities only.

The optimizer uses one-land descent and a two-land completion check. Budget exhaustion is reported explicitly and does not certify local optimality. The model does not simulate sequencing, life totals or Cavern restrictions.

In [ ]:
from pathlib import Path
import sys
from collections import defaultdict
from html import escape
from IPython.display import HTML, display

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'mana_search.py').exists())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from examples.sultai import LAND_NAMES, MANA_BASE, SPELLS
from examples.sultai_optimize import (
    LAND_COUNT, COLORS, FORMAT, LAND_FAMILIES, FIXED_LANDS, TURNS, POSITION,
    OPENING_HAND_SIZE, EVALUATION_BUDGETS, optimize,
)
from mana_lands import ManaSource
from mana_search import ManaSpace, LocalManaEvaluator, local_search

def name(source):
    return LAND_NAMES.get(source, f'{source.kind}:{source.colors}')

def lands(entries):
    return ', '.join(f'{n} {name(s)}' for s, n in entries) or 'none'

def show(headers, rows):
    head = ''.join(f'<th>{escape(str(h))}</th>' for h in headers)
    body = ''.join('<tr>' + ''.join(f'<td>{escape(str(v))}</td>' for v in row) + '</tr>' for row in rows)
    display(HTML(f'<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>'))

## Search space and structured evaluation

Per-source and aggregate family bounds belong to ManaSpace. The initial base is validated before searching. Opportunities retain cost, turn and play/draw axes. Change constraints before constructing the evaluator.

In [ ]:
from examples.sultai import MANA_MODEL

INITIAL_BASE = dict(MANA_BASE)
space = ManaSpace.from_colors(
    LAND_COUNT, COLORS, families=LAND_FAMILIES, format=FORMAT, fixed=FIXED_LANDS,
    family_bounds={'basic': (0, 2), 'shock': (0, 12), 'town': (4, 4)},
)
space.counts(INITIAL_BASE)
evaluator = LocalManaEvaluator(space, SPELLS, model=MANA_MODEL, turns=TURNS, position=POSITION,
                               opening_hand_size=OPENING_HAND_SIZE)
show(['Source', 'Minimum', 'Maximum', 'Initial'],
     [(name(s), lo, hi, INITIAL_BASE.get(s, 0)) for s, (lo, hi) in space.bounds.items()])
show(['Family', 'Minimum', 'Maximum'], [(kind, lo, hi) for kind, (lo, hi) in space.family_bounds.items()])

## Bounded optimization

Edit the visit budgets below. Both search phases share each run's budget. A completed two-land check certifies only local optimality within the supplied space.

In [ ]:
EVALUATION_BUDGETS = (200, 150, 150)
cached_before = len(evaluator.cache)
runs, winner = optimize(
    evaluator, budgets=EVALUATION_BUDGETS,
    progress=lambda i, n, best: print(
        f'Start {i+1}: {n:4d} visits' + (f'; best {float(best.score):.9f}' if best else '')),
)
baseline = evaluator.value(INITIAL_BASE)
print(f'{len(evaluator.cache) - cached_before} new evaluations')
show(['Start', 'Budget', 'Visits', 'Stop', 'Best score'],
     [(i+1, b, r.evaluations, r.stopped, float(r.best.score)) for i, (b, r) in enumerate(zip(EVALUATION_BUDGETS, runs))])
show(['Base', 'Score', 'Delta'],
     [('Initial', float(baseline.score), 0),
      ('Best found', float(winner.score), float(winner.score-baseline.score))])
show(['Source', 'Initial', 'Result', 'Change'],
     [(name(s), a, b, b-a) for s, a, b in zip(space.sources, baseline.counts, winner.counts)])

## Opportunity changes

Contributions include spell importance and normalized turn weights. Their totals reproduce the single opportunity score. Physical copies and casting-mode shares remain separate from reporting groups.

In [ ]:
before = evaluator.components(INITIAL_BASE)
after = evaluator.components(dict(zip(space.sources, winner.counts)))
positions = 2 if POSITION == 'both' else 1

def mana_value(cost):
    return int(cost.rstrip('WUBRG') or 0) + sum(cost.count(c) for c in 'WUBRG')

def comparison(key, headers):
    groups = defaultdict(lambda: [0.0, 0.0, 0.0, 0.0])
    for index, components in enumerate((before, after)):
        for row in components.opportunities:
            values = groups[key(row)]
            values[index] += float(row.expected_copies * evaluator.timing.weights[row.turn-1]) / positions
            values[index + 2] += row.weighted_contribution * float(evaluator.timing.weights[row.turn-1]) / positions
    show(headers + ['Initial copies', 'Result copies', 'Copy delta', 'Weighted delta'],
         [(*k, f'{v[0]:.8f}', f'{v[1]:.8f}', f'{v[1]-v[0]:+.8f}', f'{v[3]-v[2]:+.8f}')
          for k, v in sorted(groups.items())])

comparison(lambda r: (mana_value(r.cost), r.cost), ['MV', 'Cost'])
comparison(lambda r: (mana_value(r.cost),), ['MV'])
comparison(lambda r: (r.turn, r.position), ['Turn', 'Position'])
assert winner.score >= baseline.score
assert sum(r.evaluations for r in runs) <= sum(EVALUATION_BUDGETS)
assert abs(sum(r.weighted_contribution * float(evaluator.timing.weights[r.turn-1]) for r in after.opportunities) / positions
           - float(winner.score)) < 1e-12